# 📚 Asistente de Investigación Multiagente
Este notebook permite cargar archivos PDF y hacer preguntas o tareas de investigación.
- Utiliza un sistema RAG con agentes para resumen, escritura e inferencia de referencias IEEE.
- Puedes cargar un archivo PDF y escribir un prompt de investigación.

In [1]:
# ✅ Cargar archivos necesarios
from langgraph_app.graph_builder import compiled_graph
from vector_storage.pdf_ingestor import ingest_pdf
import os

In [2]:
# ✅ Función principal
def ejecutar_asistente(prompt: str, pdf_path: str = None):
    if pdf_path:
        if not os.path.exists(pdf_path) or not pdf_path.endswith('.pdf'):
            print(f"❌ El archivo no existe o no es un PDF: {pdf_path}")
            return
        print(f"📄 Ingestando documento: {pdf_path}")
        ingest_pdf(pdf_path)
        print("✅ Ingesta completada.\n")

    print(f"📝 Prompt: {prompt}")
    print("🚀 Ejecutando agentes...\n")

    inputs = {"prompt": prompt}
    for output in compiled_graph.stream(inputs):
        for key, value in output.items():
            if key != "intermediate_steps":
                print(f"\n🎯 Resultado final:\n{value}")

## 📎 Subir archivo PDF desde el navegador

In [3]:
from IPython.display import display
from ipywidgets import FileUpload

uploader = FileUpload(accept='.pdf', multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload')

In [4]:
if uploader.value:
    for archivo in uploader.value:
        nombre = archivo['name']
        contenido = archivo['content']
        ruta = f"data/raw_papers/{nombre}"

        # Guardar archivo
        with open(ruta, "wb") as f:
            f.write(contenido)
        print(f"✅ Guardado: {ruta}")

## 🧪 Ejecutar el asistente

In [11]:
import sys
import os
import webbrowser

sys.path.append(os.path.abspath("."))  # Asegura que se encuentra el módulo

from main import ejecutar_asistente

# Llamada de ejemplo
result_path = "data/results"
os.makedirs(result_path, exist_ok=True)  # 🔧 Crea la carpeta si no existe

pdf_path = "data/raw_papers/public_Scrum-Guide-US.pdf"
prompt = "Necesito un resumen de los puntos clave de todo este texto, tambien que me des las referencias y que me crees una introduccion para mi investigación"

respuesta = ejecutar_asistente(prompt, pdf_path)
resultado_texto = respuesta.get("result", "⚠️ No se obtuvo una respuesta.")

# Crear archivo .txt con mismo nombre base que el PDF
base_name = os.path.splitext(os.path.basename(pdf_path))[0]
output_path = os.path.join(result_path, f"{base_name}_resultado.txt")

with open(output_path, "w", encoding="utf-8") as file:
    file.write(resultado_texto)

# Abrir el archivo (en bloc de notas o visor predeterminado)
abs_path = os.path.abspath(output_path)
webbrowser.open(f"file://{abs_path}")

print(f"✅ Resultado guardado y abierto: {output_path}")


📄 Ingestando PDF: data/raw_papers/public_Scrum-Guide-US.pdf
Directorios removidos
📂 El PDF ya está en data/raw_papers/public_Scrum-Guide-US.pdf, no se copia.
📄 Cargadas 16 páginas del PDF.
✂️ Fragmentados en 54 fragmentos de texto.
📦 Vector store creado desde cero.
✅ Vector store actualizado con éxito.
✅ PDF procesado.

🧠 Ejecutando agentes con el prompt:
Necesito un resumen de los puntos clave de todo este texto, tambien que me des las referencias y que me crees una introduccion para mi investigación

📌 Nodo DECIDER seleccionó: ['resumen', 'escritura', 'referencias']
🔹 Nodo RESUMEN ejecutado
🟩 Nodo ESCRITURA ejecutado
🟡 Nodo REFERENCIAS ejecutado
✅ Resultado guardado y abierto: data/results/public_Scrum-Guide-US_resultado.txt


In [8]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
import pandas as pd

# Cargar el vector store
vector_store_path = "data/processed"
vector_store = FAISS.load_local(vector_store_path, OpenAIEmbeddings(), allow_dangerous_deserialization=True)

# Extraer los fragmentos y sus metadatos
fragmentos = []
for key, doc in vector_store.docstore._dict.items():
    fragmentos.append({
        "ID": key,
        "Contenido": doc.page_content,
        "Fuente": doc.metadata.get("source", ""),
        "Página": doc.metadata.get("page", "")
    })

# Mostrar los datos
df_fragmentos = pd.DataFrame(fragmentos)
df_fragmentos.head()  # Muestra los primeros fragmentos


,ID,Contenido,Fuente,Página
0,1f816513-378b-411d-9f76-66de0b8b73b8,The Scrum Guide™ \nThe Definitive Guide to Scr...,data/raw_papers/public_Scrum-Guide-US.pdf,0
1,b11763b8-2e33-4eca-b85b-f1a495792d82,"© 1991-2014 Ken Schwaber and Jeff Sutherland, ...",data/raw_papers/public_Scrum-Guide-US.pdf,1
2,58cbbf24-40b2-4392-85ad-bd016c323f27,The Development Team ............................,data/raw_papers/public_Scrum-Guide-US.pdf,1
3,a034151f-23c7-4311-af64-56336be6c7b2,Daily Scrum .....................................,data/raw_papers/public_Scrum-Guide-US.pdf,1
4,440c191e-187b-4f2e-a114-4bd26cb8d7d6,Sprint Backlog ..................................,data/raw_papers/public_Scrum-Guide-US.pdf,1


In [9]:
df_fragmentos["Fuente"].value_counts()


Fuente
data/raw_papers/public_Scrum-Guide-US.pdf    54
Name: count, dtype: int64

In [10]:
from PyPDF2 import PdfReader

reader = PdfReader("data/raw_papers/public_Scrum-Guide-US.pdf")
paginas_con_texto = [i for i, p in enumerate(reader.pages) if p.extract_text().strip()]
print(f"Páginas con texto: {len(paginas_con_texto)} de {len(reader.pages)}")


Páginas con texto: 16 de 16
